## Imports
Loading the required libraries: `pathlib` for path management, `pandas` for data manipulation, `numpy` for numerical operations.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

## Building the PAC reference file (`static_pac.csv`)
Loads `Static_2025.xlsx` containing information for all firms (ISIN, name, country, region). Filters only firms classified under the **PAC** (Pacific) region according to the reference file — regardless of the ISIN prefix. Saves the result to `static_pac.csv`.

In [ ]:
base = Path.cwd().parent

static = pd.read_excel(base / "data" / "raw" / "Static_2025.xlsx")

static_pac = static[static["Region"] == "PAC"][["ISIN", "NAME", "Country"]]

print(f"Number of PAC firms: {len(static_pac)}")

static_pac.to_csv(base / "data" / "processed" / "static_pac.csv", index=False)

print("static_pac.csv created")

## Cleaning CO2 Scope 1 emissions (`co2_scope1.csv`)
Loads CO2 Scope 1 emissions from `DS_CO2_SCOPE_1_Y_2025.xlsx`. Filters PAC firms by cross-referencing with `static_pac.csv`. Excludes firms with more than 50% missing values over the years 2010–2024. Transposes the result (rows = years, columns = ISIN) and saves to `co2_scope1.csv`.

In [7]:
base = Path.cwd().parent

static_pac = pd.read_csv(base / "data" / "processed" / "static_pac.csv")
pacific_isins = static_pac["ISIN"].tolist()

file_path = base / "data" / "raw" / "DS_CO2_SCOPE_1_Y_2025.xlsx"
df = pd.read_excel(file_path)

df_pacific = df[df["ISIN"].isin(pacific_isins)]

years = list(range(2010, 2025))

df_inter = df_pacific[
    df_pacific[years].isna().mean(axis=1) <= 0.5
]

df_final = df_inter.drop(columns=["NAME"]).set_index("ISIN")[years].T

print(f"Shape: {df_final.shape}")

df_final.to_csv(base / "data" / "processed" / "co2_scope1.csv")

## Cleaning prices and computing returns (`prices_clean.csv`, `returns.csv`)
Loads monthly prices from `DS_RI_T_USD_M_2025.xlsx`. Filters PAC firms via `static_pac.csv`. Applies the following cleaning steps:
- Converts Datastream error strings (`$$ER:...`) to NaN
- Excludes firms with more than 50% missing data
- Replaces prices below 0.5 with NaN (low prices)

Computes simple returns (Rₜ = Pₜ/Pₜ₋₁ − 1). Handles delisting: assigns a return of **-100%** at the first month after the last valid price, and NaN thereafter. Saves cleaned prices and returns to `data/processed/`.

In [ ]:
base = Path.cwd().parent

static_pac = pd.read_csv(base / "data" / "processed" / "static_pac.csv")
pacific_isins = static_pac["ISIN"].tolist()

file_path = base / "data" / "raw" / "DS_RI_T_USD_M_2025.xlsx"
df = pd.read_excel(file_path)

date_cols = [c for c in df.columns if c not in ['NAME', 'ISIN']]

df[date_cols] = df[date_cols].apply(pd.to_numeric, errors='coerce')

df_pacific = df[df['ISIN'].isin(pacific_isins)]

df_pacific = df_pacific[
    df_pacific[date_cols].isna().mean(axis=1) <= 0.5
]

df_pacific[date_cols] = df_pacific[date_cols].where(df_pacific[date_cols] >= 0.5, np.nan)

prices = df_pacific.set_index("ISIN")[date_cols].T
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()

returns = prices / prices.shift(1) - 1

for isin in prices.columns:
    series = prices[isin]
    last_valid = series[series > 0].last_valid_index()
    if last_valid is not None and last_valid < prices.index[-1]:
        next_pos = prices.index.get_loc(last_valid) + 1
        delist_date = prices.index[next_pos]
        returns.loc[delist_date, isin] = -1.0
        returns.loc[returns.index > delist_date, isin] = np.nan

print(f"Number of PAC firms: {prices.shape[1]}")

out = base / "data" / "processed"
out.mkdir(parents=True, exist_ok=True)

prices.to_csv(out / "prices_clean.csv")
returns.to_csv(out / "returns.csv")

print("prices_clean.csv and returns.csv created")

## Computing covariance matrix

In [4]:
base = Path.cwd().parent
df = pd.read_csv(base /'data/processed/returns.csv', index_col=0, parse_dates=True)

window_data = df.iloc[-144:]


window_data = window_data.loc[:, window_data.count() >= 36]

cov_matrix = window_data.cov() * (144 - 1) / 144

cov_matrix.to_csv(base / "data" / "processed" / "covariance_matrix.csv")

print("covariance_matrix.csv créé")

covariance_matrix.csv créé


## Converting risk-free rate excel file to csv

In [15]:
base = Path.cwd().parent

df = pd.read_excel(base / "data" / "raw" / "Risk_Free_Rate_2025.xlsx")

df.columns = ["Date", "RF"]


df.to_csv(base / "data" / "processed" / "risk_free_rate.csv")

print(f"risk_free_rate.csv created")

risk_free_rate.csv created
